In [30]:
import os

os.environ["CALITP_BQ_MAX_BYTES"] = str(400_000_000_000)

import datetime as dt  # noqa: E402
import sys  # noqa: E402
from functools import cache  # noqa: E402

from calitp_data_analysis.gcs_pandas import GCSPandas  # noqa: E402
from loguru import logger  # noqa: E402
from shared_utils import gtfs_utils_v2  # noqa: E402
from update_vars import COMPILED_CACHED_VIEWS, GTFS_DATA_DICT  # noqa: E402
import pandas as pd

from shared_utils import DBSession, schedule_rt_utils  # noqa: E402


# Fix Missing Santa Cruz Feed

A MTC 511 subfeed exists, but shouldn't be used. This situation is not considered in our current logic, which prefers subfeeds.

In [24]:
@cache
def gcs_pandas():
    return GCSPandas()


def get_operators(analysis_date: str):
    """
    Operators to download: favor subfeeds over combined regional feed.
    """
    all_operators = gtfs_utils_v2.schedule_daily_feed_to_gtfs_dataset_name(
        selected_date=analysis_date, keep_cols=None, get_df=True, feed_option="use_subfeeds"
    ).rename(columns={"gtfs_dataset_name": "name"})

    # patch until a better fix -- need to use a subfeed for Santa Cruz
    santa_cruz = gtfs_utils_v2.schedule_daily_feed_to_gtfs_dataset_name(
        selected_date=analysis_date, keep_cols=None, get_df=True, feed_option="include_precursor"
    ).rename(columns={"gtfs_dataset_name": "name"}).query('name.str.contains("Santa Cruz")')
    assert len(santa_cruz) == 1, 'query should return exactly one Santa Cruz schedule feed'

    keep_cols = ["feed_key", "name"]

    operators_to_include = pd.concat([all_operators, santa_cruz])[keep_cols]

    # There shouldn't be any duplicates by name, since we got rid
    # of precursor feeds. But, just in case, don't allow dup names.
    operators_to_include = operators_to_include.drop_duplicates(subset="name").reset_index(drop=True)

    return operators_to_include

In [25]:
from update_vars import analysis_date_list

In [26]:
df = get_operators(analysis_date_list[0])

In [27]:
df.query('name.str.contains("Union")') # missing in speeds but present here

,feed_key,name
39,fc2ea46aa5a43afe7cc24f44d8bd3eb4,Union City Flex
107,962a9feb664035676c5de19595fe6426,Bay Area 511 Union City Transit Schedule


In [28]:
df.query('name.str.contains("Santa Cruz")') # missing

,feed_key,name
239,84ff59aed2a4c130c2b4a59e89929d05,Santa Cruz Schedule


In [15]:
df = gtfs_utils_v2.schedule_daily_feed_to_gtfs_dataset_name(
    selected_date=analysis_date_list[0], keep_cols=None, get_df=True, feed_option="include_precursor"
).rename(columns={"gtfs_dataset_name": "name"})

In [18]:
df = df.query('name.str.contains("Santa Cruz")') # missing

In [19]:
len(df)

1

## vp

In [35]:
from update_vars import analysis_date_list

# from dask.distributed import Client
# client = Client("dask-scheduler.dask.svc.cluster.local:8786")

LOG_FILE = "./logs/download_vp_v2.log"

logger.add(LOG_FILE, retention="3 months")
logger.add(sys.stderr, format="{time:YYYY-MM-DD at HH:mm:ss} | {level} | {message}", level="INFO")

# Get rt_datasets that are available for that day
rt_datasets = schedule_rt_utils.filter_dim_gtfs_datasets(
    keep_cols=["key", "name", "type", "regional_feed_type"],
    custom_filtering={"type": ["vehicle_positions"]},
    get_df=True,
)

rt_datasets = rt_datasets.rename(columns={"gtfs_dataset_name": "name"})



In [36]:
rt_datasets

,gtfs_dataset_key,name,type,regional_feed_type
0,75807bd4e46ac489ca2f0bd9cdf8d449,Avalon Vehicle Positions,vehicle_positions,None
1,fb6c9155e3abf210e64bba88b8c6cb1e,Yolobus Vehicle Positions,vehicle_positions,None
2,c0459e06e71ad266d9cf3f656805f0c8,Yolobus Vehicle Positions,vehicle_positions,None
3,41772cebeabe9bdadf693a9277155361,Bay Area 511 Tri Delta VehiclePositions,vehicle_positions,None
4,f57484589efbd31f9ab17b9ee0ec9be9,Bay Area 511 Tri Delta VehiclePositions,vehicle_positions,None
...,...,...,...,...
1234,cbf03e0fef23e0d91de2ec721acf4863,SolTrans Vehicle Positions,vehicle_positions,None
1235,ed3300aa72ad4b3cc4262275d35d876e,SolTrans Vehicle Positions,vehicle_positions,Regional Precursor Feed
1236,908dfdd4ad12b304d1487b9bd3dad2a2,Santa Maria Vehicle Positions,vehicle_positions,None
1237,ed4760f268401882a242900d602255d3,Santa Maria Vehicle Positions,vehicle_positions,None


In [41]:
# Exclude regional feed and precursors
exclude = ["Bay Area 511 Regional VehiclePositions"]
rt_datasets = rt_datasets[
    ~(rt_datasets.name.isin(exclude)) & ((rt_datasets.regional_feed_type != "Regional Precursor Feed") | rt_datasets.name.str.contains("Santa Cruz"))
].reset_index(drop=True)

rt_dataset_names = rt_datasets.name.unique().tolist()
batches = determine_batches(rt_dataset_names)

In [43]:
rt_datasets.query('name.str.contains("Union")')

,gtfs_dataset_key,name,type,regional_feed_type
162,53e069ace150cfaa714356c820719b60,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,None
163,e4d9e28dbb464da5b7641a15144175d1,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,None
164,77ca561d204e0e7e15ea9c79f783971f,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,None
165,a0c13ba8814fef779dea8e569e97584d,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,None
166,9ece86772d7230ca65c6f03734105d74,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,None
167,1a4d88ca2c1b463ee081ec38d53dd5cb,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,None
217,b0fdd49bf322e6063a7366ad20529326,Union City Vehicle Positions,vehicle_positions,None
218,0ed6b6d35c136ec6254435c1917fa7de,Union City Vehicle Positions,vehicle_positions,None
640,438e9ee61c9104573c64e1658234fe26,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,Regional Subfeed
641,10a179fd35399c998ffa32b837a0ea9a,Bay Area 511 Union City Transit VehiclePositions,vehicle_positions,Regional Subfeed


In [32]:
def determine_batches(rt_names: list) -> dict:
    # https://stackoverflow.com/questions/4843158/how-to-check-if-a-string-is-a-substring-of-items-in-a-list-of-strings
    la_metro_names = [
        "LA Metro Bus",
        "LA Metro Rail",
    ]

    bay_area_large_names = ["AC Transit", "Muni"]

    bay_area_names = ["Bay Area 511"]

    # If any of the large operator name substring is
    # found in our list of names, grab those
    # be flexible bc "Vehicle Positions" and "VehiclePositions" present
    matching1 = [i for i in rt_names if any(name in i for name in la_metro_names)]

    matching2 = [i for i in rt_names if any(name in i for name in bay_area_large_names)]

    remaining_bay_area = [
        i
        for i in rt_names
        if any(name in i for name in bay_area_names) and (i not in matching1) and (i not in matching2)
    ]
    remaining = [i for i in rt_names if (i not in matching1) and (i not in matching2) and (i not in remaining_bay_area)]

    # Batch large operators together and run remaining in 2nd query
    batch_dict = {}

    batch_dict[0] = matching1
    batch_dict[1] = remaining_bay_area
    batch_dict[2] = remaining
    batch_dict[3] = matching2

    return batch_dict